In [1]:
from typing import List
import numpy as np
import time
import matplotlib.pyplot as plt
import torch

import quairkit as qkit
from quairkit import Circuit, Hamiltonian
from quairkit.database import *
from quairkit.loss import *

from quairkit.qinfo import *
from quairkit import *


In [2]:
qkit.set_seed(7)

In [3]:
# Noisy Layer ansatz
def complex_entangled_layer_by_hand(num_qubits, depth, noisy_level):
    cir = Circuit(num_qubits)

    cir.depolarizing(0.2, 1)

    for d in range(depth):
        cir.rx([1, 2, 3])
        cir.depolarizing(noisy_level[0], [1,2,3])
        cir.rz([1, 2, 3])
        cir.depolarizing(noisy_level[0], [1, 2, 3])

        cir.cx([1,2])
        cir.generalized_depolarizing(noisy_level[1], [1, 2])
        cir.cx([2,3])
        cir.generalized_depolarizing(noisy_level[1], [2, 3])
        
    return cir


In [4]:
def loss_fcn(cir1, cir2, budget) -> torch.Tensor:
    state =  nkron(bell_state(2), zero_state(2))

    c1 = (budget-1)/2 + 1
    c2 = (budget-1)/2

    output_state1 = cir1(state).trace([2,3])
    output_state2 = cir2(state).trace([2,3])

    ave_state = c1 * output_state1.density_matrix - c2*output_state2.density_matrix

    # print(trace(ave_state @ bell_state(2).density_matrix))
    cost = 1 + trace(ave_state @ ave_state) - 2*(trace(ave_state @ bell_state(2).density_matrix))
    return torch.real(cost)

In [5]:
def train_model(num_itr, LR, num_qubits, depth, budget):
    r"""Train the QNN to find the ground-state energy of the Hamiltonian H.

    Args:
        num_itr: number of training iterations
        LR: learning rate
        num_qubits: number of qubits in the quantum circuit
        depth: number of training layers in the circuit
        H: the Hamiltonian to be minimized
    """

    # noisy_level = np.array([0.0, 0.0])
    noisy_level = np.array([0.01, 0.02])


    cir1 = complex_entangled_layer_by_hand(num_qubits, depth, noisy_level)
    cir2 = complex_entangled_layer_by_hand(num_qubits, depth, noisy_level)

    loss_list, time_list = [], []

    opt = torch.optim.Adam(lr=LR, params=list(cir1.parameters()) + list(cir2.parameters())) # cir is a Circuit type
    # opt = torch.optim.SGD(lr=LR, params=list(cir1.parameters()) + list(cir2.parameters())) # cir is a Circuit type
    # opt = torch.optim.AdamW(lr=LR, params=list(cir1.parameters()) + list(cir2.parameters())) # cir is a Circuit type
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5) # activate scheduler

    for itr in range(num_itr):
        start_time = time.time()
        opt.zero_grad()

        loss = loss_fcn(cir1, cir2, budget) # compute loss

        loss.backward()
        opt.step()

        loss = loss.item()
        scheduler.step(loss) # activate scheduler

        loss_list.append(loss)
        time_list.append(time.time() - start_time)

        if itr % (num_itr // 10) == 0 or itr == num_itr - 1:
            print(f"iter: {str(itr).zfill(len(str(num_itr)))}, " +
                f"loss: {loss:.8f}, " + 
                f"lr: {scheduler.get_last_lr()[0]:.2E}, avg_time: {np.mean(time_list):.4f}s")
            time_list = []
    return loss
    

In [6]:
num_qubits = 4
depth = 5
LR, num_itr = 0.05, 600
loss_list, time_list = [], []
optimal_error_list = []
# budget = 2.5

for budget in np.linspace(1.0, 2.5, 16):
# for budget in np.array([1.9]):
    
    cost = train_model(num_itr, LR, num_qubits, depth, budget)
    print('budget=', budget, 'error', cost)
    optimal_error_list.append(cost)
    
print(optimal_error_list)

iter: 000, loss: 0.60048127, lr: 5.00E-02, avg_time: 0.0732s
iter: 060, loss: 0.13730550, lr: 5.00E-02, avg_time: 0.0466s
iter: 120, loss: 0.13227010, lr: 5.00E-02, avg_time: 0.0482s
iter: 180, loss: 0.12832355, lr: 5.00E-02, avg_time: 0.0574s
iter: 240, loss: 0.12813520, lr: 1.25E-02, avg_time: 0.0467s
iter: 300, loss: 0.12813473, lr: 1.95E-04, avg_time: 0.0468s
iter: 360, loss: 0.12813497, lr: 6.10E-06, avg_time: 0.0450s
iter: 420, loss: 0.12813509, lr: 9.54E-08, avg_time: 0.0451s
iter: 480, loss: 0.12813509, lr: 1.19E-08, avg_time: 0.0443s
iter: 540, loss: 0.12813509, lr: 1.19E-08, avg_time: 0.0455s
iter: 599, loss: 0.12813509, lr: 1.19E-08, avg_time: 0.0493s
budget= 1.0 error 0.128135085105896
iter: 000, loss: 1.09912848, lr: 5.00E-02, avg_time: 0.0501s
iter: 060, loss: 0.12091959, lr: 5.00E-02, avg_time: 0.0447s
iter: 120, loss: 0.10895395, lr: 5.00E-02, avg_time: 0.0472s
iter: 180, loss: 0.10534692, lr: 5.00E-02, avg_time: 0.0516s
iter: 240, loss: 0.10489917, lr: 5.00E-02, avg_ti